# PRISM - H-Optimus-0 + CRC
# Pathology Reliability In Scarce-label Medicine

In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive
from torchvision import transforms

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/h_optimus_0/crc'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn scipy tqdm -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

Login OK


In [3]:
import timm
model = timm.create_model(
    "hf-hub:bioptimus/H-optimus-0",
    pretrained=True, init_values=1e-5, dynamic_img_size=True
)
model = model.cuda().eval()
print("H-Optimus-0 loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/447 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

H-Optimus-0 loaded!
GPU memory: 4.58 GB


In [4]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517)),
])

In [6]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

CRC_DIR = '/content/drive/MyDrive/PRISM/datasets/crc'

# NCT-CRC-HE-100K → 70% train / 15% val / 15% test
full_dataset = ImageFolder(root=f'{CRC_DIR}/NCT-CRC-HE-100K', transform=transform)

train_size = int(0.70 * len(full_dataset))
val_size   = int(0.15 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Classes: {full_dataset.classes}")
print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

Classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
Train: 70,000
Val:   15,000
Test:  15,000


In [7]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            features = model(images)
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy() if hasattr(labels, 'numpy') else np.array(labels))
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 274/274 [4:09:34<00:00, 54.65s/it]


Train: (70000, 1536)
Extracting val features...


Extracting: 100%|██████████| 59/59 [53:31<00:00, 54.43s/it]


Val: (15000, 1536)
Extracting test features...


Extracting: 100%|██████████| 59/59 [53:02<00:00, 53.94s/it]

Test: (15000, 1536)


In [8]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy',   train_labels)
np.save(f'{EMBED_DIR}/val_features.npy',   val_features)
np.save(f'{EMBED_DIR}/val_labels.npy',     val_labels)
np.save(f'{EMBED_DIR}/test_features.npy',  test_features)
np.save(f'{EMBED_DIR}/test_labels.npy',    test_labels)
print(f"Embeddings saved to: {EMBED_DIR}")

Embeddings saved to: /content/drive/MyDrive/PRISM/embeddings/h_optimus_0/crc


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece_multiclass(y_true, y_prob, n_bins=15):
    confidence  = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    correct     = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correct[mask].mean() - confidence[mask].mean())
    return ece / len(y_true)

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob    = clf.predict_proba(test_f)
    y_pred    = clf.predict(test_f)
    n_classes = y_prob.shape[1]
    return {
        'model': 'H-Optimus-0', 'dataset': 'CRC',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob, multi_class='ovr', average='macro'),
        'f1':    f1_score(test_l, y_pred, average='macro'),
        'ece':   compute_ece_multiclass(test_l, y_prob),
        'brier': np.mean([brier_score_loss((test_l==c).astype(int), y_prob[:,c]) for c in range(n_classes)]),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.9994 | ECE: 0.2310 | Brier: 0.0118
  1% | seed 123 | AUROC: 0.9994 | ECE: 0.2280 | Brier: 0.0118
  1% | seed 456 | AUROC: 0.9995 | ECE: 0.2316 | Brier: 0.0119
  5% | seed 42 | AUROC: 0.9997 | ECE: 0.0794 | Brier: 0.0036
  5% | seed 123 | AUROC: 0.9996 | ECE: 0.0789 | Brier: 0.0035
  5% | seed 456 | AUROC: 0.9997 | ECE: 0.0793 | Brier: 0.0035
  10% | seed 42 | AUROC: 0.9998 | ECE: 0.0487 | Brier: 0.0023
  10% | seed 123 | AUROC: 0.9997 | ECE: 0.0478 | Brier: 0.0023
  10% | seed 456 | AUROC: 0.9997 | ECE: 0.0489 | Brier: 0.0023
  25% | seed 42 | AUROC: 0.9999 | ECE: 0.0258 | Brier: 0.0015
  25% | seed 123 | AUROC: 0.9999 | ECE: 0.0279 | Brier: 0.0015
  25% | seed 456 | AUROC: 0.9999 | ECE: 0.0263 | Brier: 0.0015
  50% | seed 42 | AUROC: 0.9999 | ECE: 0.0165 | Brier: 0.0011
  50% | seed 123 | AUROC: 0.9999 | ECE: 0.0165 | Brier: 0.0011
  50% | seed 456 | AUROC: 0.9999 | ECE: 0.0163 | Brier: 0.0011
  100% | seed 42 | AUROC: 1.0000 | ECE: 0.0111 | Brier: 0.0008
  1

In [10]:
from scipy.optimize import minimize_scalar

def find_temperature_multiclass(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits    = clf.decision_function(val_f)
    T             = find_temperature_multiclass(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)
    ece_raw       = compute_ece_multiclass(test_l, test_prob_raw)
    test_logits   = clf.decision_function(test_f)
    scaled        = test_logits / T
    exp_s         = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s   = exp_s / exp_s.sum(axis=1, keepdims=True)
    ece_scaled    = compute_ece_multiclass(test_l, test_prob_s)
    n_classes     = test_prob_raw.shape[1]
    return {
        'model': 'H-Optimus-0', 'dataset': 'CRC',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc':       roc_auc_score(test_l, test_prob_raw, multi_class='ovr', average='macro'),
        'brier_raw':   np.mean([brier_score_loss((test_l==c).astype(int), test_prob_raw[:,c]) for c in range(n_classes)]),
        'brier_scaled':np.mean([brier_score_loss((test_l==c).astype(int), test_prob_s[:,c])   for c in range(n_classes)]),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           0.3154   0.2302      0.0037           0.2265
0.05           0.4759   0.0792      0.0030           0.0762
0.10           0.5280   0.0485      0.0018           0.0467
0.25           0.5832   0.0266      0.0018           0.0248
0.50           0.6062   0.0164      0.0014           0.0150
1.00           0.6212   0.0111      0.0012           0.0099


In [11]:
df.to_csv(f'{RESULTS_DIR}/h_optimus_0_crc_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/h_optimus_0_crc_temperature_scaling.csv', index=False)
print("Results saved!")
print(f"  {RESULTS_DIR}/h_optimus_0_crc_results.csv")
print(f"  {RESULTS_DIR}/h_optimus_0_crc_temperature_scaling.csv")

Results saved!
  /content/drive/MyDrive/PRISM/results/h_optimus_0_crc_results.csv
  /content/drive/MyDrive/PRISM/results/h_optimus_0_crc_temperature_scaling.csv
